# Olist Data Cleaning Notebook
## Author: Ahmed Mohamed Awadalla
## Date: May 2026

This notebook cleans the raw Olist CSV files and exports them as analysis-ready datasets.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")

## 2. Define File Paths

In [ ]:
# Set your raw data folder path here
raw_data_path = "../data/raw/"  # Change this to your path
clean_data_path = "../data/cleaned/"

# Create cleaned folder if not exists
os.makedirs(clean_data_path, exist_ok=True)

print(f"Raw data path: {raw_data_path}")
print(f"Clean data path: {clean_data_path}")

## 3. Load All CSV Files

In [ ]:
# Load all 9 CSV files
orders = pd.read_csv(raw_data_path + "olist_orders_dataset.csv")
customers = pd.read_csv(raw_data_path + "olist_customers_dataset.csv")
products = pd.read_csv(raw_data_path + "olist_products_dataset.csv")
sellers = pd.read_csv(raw_data_path + "olist_sellers_dataset.csv")
order_items = pd.read_csv(raw_data_path + "olist_order_items_dataset.csv")
payments = pd.read_csv(raw_data_path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(raw_data_path + "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(raw_data_path + "olist_geolocation_dataset.csv")
translation = pd.read_csv(raw_data_path + "product_category_name_translation.csv")

print("All files loaded successfully")
print(f"Orders: {orders.shape}")
print(f"Customers: {customers.shape}")
print(f"Products: {products.shape}")
print(f"Sellers: {sellers.shape}")
print(f"Order Items: {order_items.shape}")
print(f"Payments: {payments.shape}")
print(f"Reviews: {reviews.shape}")
print(f"Geolocation: {geolocation.shape}")
print(f"Translation: {translation.shape}")

## 4. Explore Data Structure

In [ ]:
# Check orders table
print("=== ORDERS ===")
print(orders.info())
print("\nNull values in orders:")
print(orders.isnull().sum())
print("\nSample data:")
print(orders.head())

In [ ]:
# Check products table
print("=== PRODUCTS ===")
print(products.info())
print("\nNull values in products:")
print(products.isnull().sum())

## 5. Handle Missing Values

In [ ]:
# 5.1 Handle missing product categories - fill with 'Unknown'
products['product_category_name'] = products['product_category_name'].fillna('Unknown')

# 5.2 Handle missing product dimensions - fill with median
dimension_cols = ['product_name_length', 'product_description_length', 
                  'product_photos_qty', 'product_weight_g', 
                  'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in dimension_cols:
    if col in products.columns:
        median_val = products[col].median()
        products[col] = products[col].fillna(median_val)
        print(f"Filled {col} with median: {median_val:.2f}")

In [ ]:
# 5.3 Handle missing review comments - fill with empty string
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')

# 5.4 Handle missing geolocation
geolocation = geolocation.dropna(subset=['geolocation_lat', 'geolocation_lng'])

print("Missing values handled successfully")

## 6. Remove Duplicates

In [ ]:
# Remove duplicates from geolocation (keep first per zip code)
geolocation = geolocation.drop_duplicates(subset=['geolocation_zip_code_prefix'], keep='first')
print(f"Geolocation after deduplication: {geolocation.shape}")

# Check for duplicates in other tables
print(f"Orders duplicates: {orders.duplicated().sum()}")
print(f"Customers duplicates: {customers.duplicated().sum()}")
print(f"Products duplicates: {products.duplicated().sum()}")

## 7. Convert Date Columns to Datetime

In [ ]:
# Date columns in orders
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors='coerce')
        print(f"Converted {col}")

# Date columns in reviews
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

print("\nAll date columns converted")

## 8. Feature Engineering: Calculate Delivery Days

In [ ]:
# Calculate actual delivery days
orders['actual_delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

# Calculate difference between estimated and actual delivery
orders['delivery_diff_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

print("Delivery metrics calculated:")
print(f"Average actual delivery days: {orders['actual_delivery_days'].mean():.2f}")
print(f"Average delivery diff (negative = late): {orders['delivery_diff_days'].mean():.2f}")

## 9. Merge English Category Names

In [ ]:
# Merge translation with products
products = products.merge(
    translation, 
    on='product_category_name', 
    how='left'
)

# Fill missing English names with the Portuguese name
products['product_category_name_english'] = products['product_category_name_english'].fillna(products['product_category_name'])

print("English category names added")
print(products[['product_category_name', 'product_category_name_english']].drop_duplicates().head(10))

## 10. Export Cleaned Data

In [ ]:
# Export all cleaned tables
orders.to_csv(clean_data_path + "orders_clean.csv", index=False)
customers.to_csv(clean_data_path + "customers_clean.csv", index=False)
products.to_csv(clean_data_path + "products_clean.csv", index=False)
sellers.to_csv(clean_data_path + "sellers_clean.csv", index=False)
order_items.to_csv(clean_data_path + "order_items_clean.csv", index=False)
payments.to_csv(clean_data_path + "order_payments_clean.csv", index=False)
reviews.to_csv(clean_data_path + "order_reviews_clean.csv", index=False)
geolocation.to_csv(clean_data_path + "geolocation_clean.csv", index=False)

print("All cleaned files exported successfully to:", clean_data_path)
print("\nExported files:")
print("- orders_clean.csv")
print("- customers_clean.csv")
print("- products_clean.csv")
print("- sellers_clean.csv")
print("- order_items_clean.csv")
print("- order_payments_clean.csv")
print("- order_reviews_clean.csv")
print("- geolocation_clean.csv")

## 11. Final Summary

In [ ]:
print("=" * 50)
print("DATA CLEANING SUMMARY")
print("=" * 50)
print(f"\nOrders: {len(orders)} rows, {len(orders.columns)} columns")
print(f"Customers: {len(customers)} rows, {len(customers.columns)} columns")
print(f"Products: {len(products)} rows, {len(products.columns)} columns")
print(f"Sellers: {len(sellers)} rows, {len(sellers.columns)} columns")
print(f"Order Items: {len(order_items)} rows, {len(order_items.columns)} columns")
print(f"Payments: {len(payments)} rows, {len(payments.columns)} columns")
print(f"Reviews: {len(reviews)} rows, {len(reviews.columns)} columns")
print(f"Geolocation: {len(geolocation)} rows, {len(geolocation.columns)} columns")

print("\n" + "=" * 50)
print("DATA CLEANING COMPLETED SUCCESSFULLY")
print("=" * 50)